# Phase 4 — Linear Model vs Ensemble: Logistic Regression vs Random Forest
## Dataset: GSE6575 — Gene Expression in Blood of Children with ASD
**Dr. Nourhène Ben Rabah — Introduction to Machine Learning**

This phase trains a **Random Forest** ensemble and compares it against a
**Logistic Regression** — a simple *linear* classifier. The two embody opposite
modelling philosophies (linear & parametric vs non-linear & non-parametric), so
comparing them shows which one best fits this small, very high-dimensional
dataset.

- **Part 1** — Principle of Logistic Regression and of Random Forest
- **Part 2** — Training and evaluation of Random Forest on the test data
- **Part 3** — Comparison with Logistic Regression
- **Part 4** — Advantages and limitations: linear model vs ensemble

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from matplotlib.colors import ListedColormap

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("All libraries loaded successfully.")

---
## Load Data and Reproduce Phase 3 Setup

We reload the same prepared data and reproduce the **exact same** train/test split
and PCA used in Phase 3, so that the comparison is fair.


In [ ]:
# ── Load prepared data (same files as Phase 3) ───────────────────────────────
X = pd.read_csv('X_prepared.csv', index_col=0)
y = pd.read_csv('y_labels.csv', index_col=0).squeeze()

print(f"Feature matrix X : {X.shape}  (samples x probes)")
print(f"Label vector y   : {y.shape}")
print()
print("Class distribution:")
for cls, n in y.value_counts().items():
    print(f"  {cls:<35} {n} samples")

In [ ]:
# ── Reproduce Phase 3 train/test split and PCA exactly ───────────────────────
# Same random_state=42, test_size=0.20, stratify=y as Phase 3.
# This ensures we compare both models on the IDENTICAL test samples.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Train / Test split  : {X_train.shape[0]} / {X_test.shape[0]} samples")
print(f"PCA components kept : {pca.n_components_}  (explaining 95% of variance)")
print(f"X_train after PCA   : {X_train_pca.shape}")
print(f"X_test  after PCA   : {X_test_pca.shape}")

---
## Part 1 — Principle: Logistic Regression and Random Forest

### 1.1 How Logistic Regression works

**Logistic Regression** is a **linear** classifier. It learns one weight
(coefficient) per feature, combines them linearly, and passes the result through a
logistic/softmax function to obtain class probabilities. The decision boundary it
draws is therefore a **hyperplane** — a flat boundary in feature space.

Key consequences:
- It is **parametric**: the whole model is just a vector of coefficients per class.
- It has **higher bias but lower variance** than a tree ensemble — it cannot
  capture non-linear interactions, but it is very stable on small datasets.
- It is **interpretable**: each coefficient says how a feature pushes the
  prediction toward a class.
- It is **scale-sensitive** (because of the L2 penalty), so the PCA components are
  **standardised** before fitting — handled inside the model pipeline in Part 3.

### 1.2 How Random Forest works

Random Forest is a **non-linear, non-parametric ensemble** that builds many
decision trees and combines them by **majority vote**. Two sources of randomness
decorrelate the trees:

```
For each tree t = 1 ... n_estimators:
    1. Draw a bootstrap sample (n samples with replacement) from training data
    2. At each node:
       a. Draw max_features random features
       b. Find the BEST split threshold among those candidates (Gini gain)
    3. Grow the tree until min_samples_leaf / max_depth is reached
Final prediction = majority vote across all trees
```

Because each tree splits the space into axis-aligned regions, the forest can fit
**non-linear** boundaries and feature interactions that a single hyperplane
cannot.

### 1.3 Logistic Regression vs Random Forest — key differences

| Property | Logistic Regression | Random Forest |
|---|---|---|
| **Model family** | Linear, parametric | Non-linear, non-parametric (ensemble) |
| **Decision boundary** | Single hyperplane | Many axis-aligned regions |
| **Captures interactions** | ❌ No (unless added manually) | ✅ Yes |
| **Bias / variance** | Higher bias, lower variance | Lower bias, higher variance |
| **Feature scaling** | Required (L2 penalty) | Not needed |
| **Interpretability** | High (coefficients) | Low (black box of 100 trees) |
| **Training speed** | Very fast | Moderate |
| **Best for** | Linearly separable / small-n | Complex non-linear structure |

### 1.4 Why compare Logistic Regression against Random Forest?

The two models sit at opposite ends of the bias–variance spectrum, which makes the
comparison informative on a `p >> n` genomics dataset (54K features, 56 samples):

- If the classes are roughly **linearly separable** in PCA space, the regularised
  linear model will match or beat the forest while being far simpler and more
  stable — a common outcome when n is tiny.
- If the structure is genuinely **non-linear**, the forest should pull ahead.

So the comparison effectively asks: *does the extra flexibility of a non-linear
ensemble actually pay off here, or is a simple linear boundary enough?*

---
## Part 2 — Training and Evaluation of Random Forest

### 2.1 Hyperparameters


In [ ]:
# ── Random Forest model ──────────────────────────────────────────────────────
# We use the same hyperparameters as Phase 3's Extra Trees for a fair comparison.
# The only difference is the algorithm itself.

rf_model = RandomForestClassifier(
    n_estimators=100,       # same as Phase 3
    max_features='sqrt',    # same as Phase 3
    class_weight='balanced',# same as Phase 3 — handles MR/DD imbalance
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_pca, y_train)
print("Random Forest model trained successfully.")
print(f"  n_estimators : {rf_model.n_estimators}")
print(f"  max_features : {rf_model.max_features}")
print(f"  n_features   : {X_train_pca.shape[1]} PCA components")

### 2.2 Training Accuracy

In [ ]:
y_train_pred_rf = rf_model.predict(X_train_pca)
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
train_f1_rf  = f1_score(y_train, y_train_pred_rf, average='weighted')

print("=== Random Forest — Training Results ===")
print(f"  Training Accuracy        : {train_acc_rf:.4f}")
print(f"  Training F1 (weighted)   : {train_f1_rf:.4f}")
print()
print("Note: Random Forest can fit the training data almost perfectly (high")
print("training accuracy). A regularised Logistic Regression fits only a linear")
print("boundary, so it usually shows a smaller train/test gap — higher bias but")
print("lower variance.")

### 2.3 Cross-Validation (5-Fold Stratified)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Pipeline: PCA inside CV to avoid data leakage (same as Phase 3)
rf_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_cv_acc = cross_val_score(rf_pipeline, X, y, cv=skf,
                             scoring='accuracy', n_jobs=-1)
rf_cv_f1  = cross_val_score(rf_pipeline, X, y, cv=skf,
                             scoring='f1_weighted', n_jobs=-1)

print("=== Random Forest — 5-Fold Cross-Validation ===")
print(f"  Accuracy per fold  : {[f'{s:.3f}' for s in rf_cv_acc]}")
print(f"  Mean Accuracy      : {rf_cv_acc.mean():.4f} +/- {rf_cv_acc.std():.4f}")
print()
print(f"  F1 per fold        : {[f'{s:.3f}' for s in rf_cv_f1]}")
print(f"  Mean F1 (weighted) : {rf_cv_f1.mean():.4f} +/- {rf_cv_f1.std():.4f}")

### 2.4 Test Set Evaluation

In [ ]:
y_pred_rf = rf_model.predict(X_test_pca)
test_acc_rf = accuracy_score(y_test, y_pred_rf)
test_f1_rf  = f1_score(y_test, y_pred_rf, average='weighted')

print("=== Random Forest — Test Set Results ===")
print(f"  Test Accuracy      : {test_acc_rf:.4f}")
print(f"  Test F1 (weighted) : {test_f1_rf:.4f}")
print(f"  Test samples       : {X_test.shape[0]}")

### 2.5 Classification Report

In [ ]:
print("=== Classification Report — Random Forest (Test Set) ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))

### 2.6 Confusion Matrix

In [ ]:
labels = sorted(y.unique())
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(
    ax=ax, cmap='Blues', colorbar=True
)
ax.set_title('Confusion Matrix — Random Forest (Test Set)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 2.7 Feature Importance — Top PCA Components

In [ ]:
importances_rf = rf_model.feature_importances_
top_n  = min(20, len(importances_rf))
top_idx = np.argsort(importances_rf)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_n), importances_rf[top_idx],
       color='mediumseagreen', edgecolor='white')
ax.set_xticks(range(top_n))
ax.set_xticklabels([f'PC{i+1}' for i in top_idx], rotation=45, ha='right')
ax.set_xlabel('PCA Component')
ax.set_ylabel('Feature Importance (Gini)')
ax.set_title('Top 20 Most Important PCA Components — Random Forest')
plt.tight_layout()
plt.show()

---
## Part 3 — Comparison: Random Forest vs Logistic Regression

We now train a **Logistic Regression** on the **identical** setup (same split,
same PCA, same class weighting). Because the linear model is scale-sensitive, a
`StandardScaler` is added inside its pipeline — the only preprocessing difference,
and one the tree ensemble does not need.

In [ ]:
# ── Train a Logistic Regression (linear baseline) ────────────────────────────
# A StandardScaler is bundled in: PCA components have very different variances and
# the L2 penalty of LogisticRegression is scale-sensitive. Random Forest, being
# tree-based, needs no scaling — so each model gets the preprocessing it requires
# while sharing the same PCA features and class weighting.
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',   # same imbalance handling as RF
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])
lr_model.fit(X_train_pca, y_train)

# Training accuracy (to compare overfitting behaviour)
y_train_pred_lr = lr_model.predict(X_train_pca)
train_acc_lr = accuracy_score(y_train, y_train_pred_lr)

# Test performance
y_pred_lr   = lr_model.predict(X_test_pca)
test_acc_lr = accuracy_score(y_test, y_pred_lr)
test_f1_lr  = f1_score(y_test, y_pred_lr, average='weighted')

# Cross-validation with the same leakage-safe pipeline (PCA + scaling inside CV)
lr_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ))
])
lr_cv_acc = cross_val_score(lr_pipeline, X, y, cv=skf,
                            scoring='accuracy', n_jobs=-1)
lr_cv_f1  = cross_val_score(lr_pipeline, X, y, cv=skf,
                            scoring='f1_weighted', n_jobs=-1)

print("Logistic Regression trained successfully.")
print(f"  Coefficients shape : {lr_model.named_steps['clf'].coef_.shape}  (classes x PCA components)")
print(f"  Training accuracy  : {train_acc_lr:.4f}  (RF train acc: {train_acc_rf:.4f})")
print(f"  Test accuracy      : {test_acc_lr:.4f}")

### 3.1 Linear vs Non-linear Decision Boundary

The clearest way to see the difference between the two models is the **shape of
their decision boundary**. Using only the first two PCA components (for a 2-D
view), we fit each model and shade the region it assigns to each class.

Logistic Regression produces **straight boundaries** (a linear model), while
Random Forest carves out **irregular, non-linear regions**. This is an
illustration on 2 components only — the real models above use all PCA components.

In [ ]:
# Illustrative 2-D fit on the first two PCA components only
X_tr2 = X_train_pca[:, :2]
classes_sorted = sorted(y_train.unique())
n_cls = len(classes_sorted)
code = {c: i for i, c in enumerate(classes_sorted)}
y_tr_codes = y_train.map(code).values

lr_2d = Pipeline([('scaler', StandardScaler()),
                  ('clf', LogisticRegression(class_weight='balanced',
                                             max_iter=1000, random_state=RANDOM_STATE))])
rf_2d = RandomForestClassifier(n_estimators=100, max_features='sqrt',
                               class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
lr_2d.fit(X_tr2, y_tr_codes)
rf_2d.fit(X_tr2, y_tr_codes)

# Mesh grid over the PC1/PC2 plane
pad = 1.0
x_min, x_max = X_tr2[:, 0].min() - pad, X_tr2[:, 0].max() + pad
y_min, y_max = X_tr2[:, 1].min() - pad, X_tr2[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

cmap = ListedColormap(list(plt.cm.Set2.colors[:n_cls]))
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, model, name in [(axes[0], lr_2d, 'Logistic Regression (linear)'),
                        (axes[1], rf_2d, 'Random Forest (non-linear)')]:
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.30, cmap=cmap, vmin=0, vmax=n_cls - 1)
    ax.scatter(X_tr2[:, 0], X_tr2[:, 1], c=y_tr_codes, cmap=cmap,
               vmin=0, vmax=n_cls - 1, edgecolor='k', s=45)
    ax.set_title(name)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=8,
                      markerfacecolor=cmap(i), markeredgecolor='k', label=c)
           for i, c in enumerate(classes_sorted)]
axes[1].legend(handles=handles, fontsize=7, loc='best')
plt.suptitle('Decision boundaries on the first two PCA components (illustrative)')
plt.tight_layout()
plt.show()

### 3.2 Metrics Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Metric': [
        'Training Accuracy',
        'CV Accuracy (mean)',
        'CV Accuracy (std)',
        'CV F1 weighted (mean)',
        'CV F1 weighted (std)',
        'Test Accuracy',
        'Test F1 weighted',
    ],
    'Logistic Regression': [
        f'{train_acc_lr:.4f}',
        f'{lr_cv_acc.mean():.4f}',
        f'{lr_cv_acc.std():.4f}',
        f'{lr_cv_f1.mean():.4f}',
        f'{lr_cv_f1.std():.4f}',
        f'{test_acc_lr:.4f}',
        f'{test_f1_lr:.4f}',
    ],
    'Random Forest': [
        f'{train_acc_rf:.4f}',
        f'{rf_cv_acc.mean():.4f}',
        f'{rf_cv_acc.std():.4f}',
        f'{rf_cv_f1.mean():.4f}',
        f'{rf_cv_f1.std():.4f}',
        f'{test_acc_rf:.4f}',
        f'{test_f1_rf:.4f}',
    ]
})

print("=== Performance Comparison: Logistic Regression vs Random Forest ===")
print(comparison.to_string(index=False))
print()
print("Training-accuracy gap (overfitting indicator):")
print(f"  Logistic Regression : {train_acc_lr:.4f} train  ->  {test_acc_lr:.4f} test"
      f"   (drop {train_acc_lr - test_acc_lr:+.4f})")
print(f"  Random Forest       : {train_acc_rf:.4f} train  ->  {test_acc_rf:.4f} test"
      f"   (drop {train_acc_rf - test_acc_rf:+.4f})")

### 3.3 Visual Comparison — CV Accuracy per Fold

In [ ]:
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(len(folds))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV accuracy per fold
axes[0].bar(x - width/2, lr_cv_acc, width, label='Logistic Regression',
            color='slateblue', edgecolor='white')
axes[0].bar(x + width/2, rf_cv_acc, width, label='Random Forest',
            color='mediumseagreen', edgecolor='white')
axes[0].axhline(lr_cv_acc.mean(), color='slateblue',
                linestyle='--', alpha=0.7, label=f'LR mean={lr_cv_acc.mean():.3f}')
axes[0].axhline(rf_cv_acc.mean(), color='mediumseagreen',
                linestyle='--', alpha=0.7, label=f'RF mean={rf_cv_acc.mean():.3f}')
axes[0].set_xticks(x)
axes[0].set_xticklabels(folds)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CV Accuracy per Fold')
axes[0].legend(fontsize=8)

# Summary bar chart
metrics   = ['CV Accuracy', 'CV F1', 'Test Accuracy', 'Test F1']
lr_values = [lr_cv_acc.mean(), lr_cv_f1.mean(), test_acc_lr, test_f1_lr]
rf_values = [rf_cv_acc.mean(), rf_cv_f1.mean(), test_acc_rf, test_f1_rf]
x2 = np.arange(len(metrics))

axes[1].bar(x2 - width/2, lr_values, width, label='Logistic Regression',
            color='slateblue', edgecolor='white')
axes[1].bar(x2 + width/2, rf_values, width, label='Random Forest',
            color='mediumseagreen', edgecolor='white')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metrics, rotation=15, ha='right')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Score')
axes[1].set_title('Overall Performance Comparison')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 3.4 Side-by-Side Confusion Matrices

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=labels).plot(
    ax=axes[0], cmap='Purples', colorbar=False)
axes[0].set_title('Logistic Regression')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Random Forest')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

### 3.5 Coefficients vs Feature Importance

In [ ]:
importances_rf = rf_model.feature_importances_                       # Gini importance
coef_lr = np.abs(lr_model.named_steps['clf'].coef_).mean(axis=0)    # mean |coef| across classes
n_components = len(importances_rf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression: mean |coefficient| per PC
top_lr = np.argsort(coef_lr)[::-1][:15]
axes[0].bar(range(15), coef_lr[top_lr], color='slateblue', edgecolor='white')
axes[0].set_xticks(range(15))
axes[0].set_xticklabels([f'PC{i+1}' for i in top_lr], rotation=45, ha='right')
axes[0].set_title('Logistic Regression — Top 15 PCs by |coefficient|')
axes[0].set_ylabel('Mean |coefficient| across classes')

# Random Forest: Gini importance per PC
top_rf = np.argsort(importances_rf)[::-1][:15]
axes[1].bar(range(15), importances_rf[top_rf], color='mediumseagreen', edgecolor='white')
axes[1].set_xticks(range(15))
axes[1].set_xticklabels([f'PC{i+1}' for i in top_rf], rotation=45, ha='right')
axes[1].set_title('Random Forest — Top 15 PCs by Gini importance')
axes[1].set_ylabel('Gini Importance')

plt.tight_layout()
plt.show()

print("Note: the two scores are not comparable in magnitude — |coefficient|")
print("measures linear influence, Gini importance measures how usefully a PC is")
print("used for splitting. We compare WHICH PCs each model relies on, not the")
print("raw values.")
print()
top5_lr = set([f'PC{i+1}' for i in top_lr[:5]])
top5_rf = set([f'PC{i+1}' for i in top_rf[:5]])
print(f"  Logistic Regression top 5 : {sorted(top5_lr)}")
print(f"  Random Forest top 5       : {sorted(top5_rf)}")
print(f"  Common PCs                : {sorted(top5_lr & top5_rf)}")

---
## Part 4 — Advantages and Limitations: Linear Model vs Ensemble

### 4.1 Linear model vs non-linear ensemble

This comparison is about **model flexibility** — the bias–variance trade-off seen
from the opposite side of a single tree:

- **Logistic Regression** is a high-bias, low-variance linear model. It can only
  draw a hyperplane, so it underfits if the classes are not linearly separable —
  but it is very stable when data is scarce, and regularisation keeps it from
  overfitting 54K-dimensional inputs.
- **Random Forest** is a low-bias, higher-variance non-linear model. It captures
  interactions a hyperplane cannot, but with only 56 samples it can overfit and
  its predictions vary more from fold to fold.
- On `p >> n` genomics problems, a regularised linear model is often a
  surprisingly strong baseline; the forest only wins if there is real non-linear
  structure to exploit.

### 4.2 Advantages

| Advantage | Logistic Regression | Random Forest |
|---|---|---|
| Captures non-linear interactions | ❌ No | ✅ Yes |
| Interpretability | ✅ High (coefficients) | ❌ Black box (100 trees) |
| Stability on small n | ✅ High (low variance) | ⚠️ Moderate |
| Training / prediction speed | ✅ Very fast | ⚠️ Slower (100 trees) |
| Handles p >> n | ✅ With regularisation | ✅ With feature subsampling |
| Handles class imbalance (`class_weight`) | ✅ Yes | ✅ Yes |
| Needs no feature scaling | ❌ Scaling required | ✅ Scale-invariant |

### 4.3 Limitations

| Limitation | Logistic Regression | Random Forest |
|---|---|---|
| Underfits non-linear data | ❌ Yes (linear only) | ✅ Handles it |
| Overfitting with small n | ✅ Resistant (regularised) | ⚠️ Can overfit |
| Sensitivity to feature scale | ❌ Sensitive | ✅ Insensitive |
| Memory usage | ✅ Low (one weight vector) | ❌ Higher (stores 100 trees) |
| Gene-level interpretation | ⚠️ Only via PCA back-projection | ⚠️ Only via PCA back-projection |

### 4.4 Which is better for this dataset?

In [ ]:
print("=== Final Comparison Summary ===")
print()
print(f"{'Metric':<30} {'Logistic Reg.':>15} {'Random Forest':>15}")
print("-" * 62)
print(f"{'Training Accuracy':<30} {train_acc_lr:>15.4f} {train_acc_rf:>15.4f}")
print(f"{'CV Accuracy (mean)':<30} {lr_cv_acc.mean():>15.4f} {rf_cv_acc.mean():>15.4f}")
print(f"{'CV Accuracy (std)':<30} {lr_cv_acc.std():>15.4f} {rf_cv_acc.std():>15.4f}")
print(f"{'CV F1 weighted (mean)':<30} {lr_cv_f1.mean():>15.4f} {rf_cv_f1.mean():>15.4f}")
print(f"{'Test Accuracy':<30} {test_acc_lr:>15.4f} {test_acc_rf:>15.4f}")
print(f"{'Test F1 weighted':<30} {test_f1_lr:>15.4f} {test_f1_rf:>15.4f}")
print()

if rf_cv_acc.mean() >= lr_cv_acc.mean():
    winner = "Random Forest"
    reason = ("The non-linear ensemble captures structure the linear boundary "
              "misses and generalises better here despite the small sample.")
else:
    winner = "Logistic Regression"
    reason = ("The regularised linear model matches or beats the forest while "
              "being simpler and lower-variance — a sign the classes are close "
              "to linearly separable in PCA space, common when n is tiny.")

print(f"Best CV performer: {winner}")
print(f"Reason: {reason}")
print()
print("Important caveat: with only 12 test samples, test-set results have very")
print("high variance. Cross-validation is the more reliable estimate here.")

### 4.5 Written Analysis

#### Performance comparison

Both models were trained under identical conditions (same PCA, same split, same
class weighting), the only difference being the `StandardScaler` that Logistic
Regression needs and Random Forest does not. This isolates the effect that
matters: a **linear** model vs a **non-linear** ensemble.

The two sit at opposite ends of the bias–variance spectrum. Logistic Regression
has higher bias but lower variance — it usually shows a small train/test gap and
stable CV folds. Random Forest has lower bias but higher variance — it can fit the
training data almost perfectly yet vary more across folds. Which one wins on the
test/CV scores tells us whether the classes are close to linearly separable in PCA
space (favouring Logistic Regression) or have genuine non-linear structure
(favouring Random Forest).

With only 56 samples the CV confidence intervals of the two models overlap, so any
gap should be read cautiously and cross-validation trusted over the 12-sample test
set.

#### Advantages of the linear model

1. **Stability on scarce data**: regularised Logistic Regression has low variance,
   which is valuable when n is tiny — it does not chase noise the way a flexible
   model can.
2. **Interpretability**: each coefficient quantifies how a PCA component pushes the
   prediction toward a class; the forest only offers aggregate importances.
3. **Speed and simplicity**: it is essentially a single weight vector, trained in a
   fraction of the time the forest needs.

#### Advantages of the ensemble

1. **Non-linear structure**: Random Forest can model interactions and curved
   boundaries that a hyperplane cannot (see the decision-boundary plot in 3.1).
2. **No scaling required**: tree splits are scale-invariant, so it is robust to the
   differing variances of PCA components.
3. **Stable feature importance**: importances are averaged over 100 trees.

#### Limitations specific to this dataset

1. **Sample-size bottleneck**: with n=56, neither model can fully exploit its
   strengths; both show high variance across CV folds.
2. **PCA information loss**: reducing 54,630 probes to ~41 PCA components (95%
   variance) discards 5% of the signal for both models.
3. **Biological interpretability**: coefficients and importances are expressed over
   PCA components, not individual genes; back-projecting through `pca.components_`
   is needed to recover specific biomarkers.

#### Conclusion

Logistic Regression and Random Forest embody two modelling philosophies. The
linear model is fast, stable and interpretable but cannot capture non-linear
structure; the forest is flexible but more prone to variance on so few samples. On
`p >> n` genomics data a regularised linear baseline is frequently hard to beat, so
the choice should be driven by cross-validation rather than the small test set. For
future work, enlarging the dataset (e.g. merging GSE6575 with GSE18123) would give
a firmer basis for deciding whether the forest's extra flexibility is justified.

In [ ]:
print("=== Phase 4 Complete ===")
print()
print("Summary:")
print(f"  Linear model   : Logistic Regression + PCA(95%) + scaling")
print(f"    Train Accuracy: {train_acc_lr:.4f}   (CV {lr_cv_acc.mean():.4f} +/- {lr_cv_acc.std():.4f})")
print(f"    Test  Accuracy: {test_acc_lr:.4f}")
print()
print(f"  Ensemble model : Random Forest + PCA(95%)")
print(f"    Train Accuracy: {train_acc_rf:.4f}   (CV {rf_cv_acc.mean():.4f} +/- {rf_cv_acc.std():.4f})")
print(f"    Test  Accuracy: {test_acc_rf:.4f}")
print()
print("Logistic Regression draws a linear boundary (higher bias, lower variance);")
print("Random Forest combines 100 trees into a non-linear boundary (lower bias,")
print("higher variance). Cross-validation decides which fits this small dataset.")